# Foundational Models Experiment

A small comparison of Gemini responses under different prompting conditions.

In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / '.venv' / 'Scripts' / 'python.exe').exists():
    project_root = project_root.parent

expected_python = (project_root / '.venv' / 'Scripts' / 'python.exe').resolve()
assert Path(sys.executable).resolve() == expected_python
assert sys.version_info[:3] == (3, 13, 1)
print('Loaded local environment; using the project .venv with Python 3.13.1.')

Loaded local environment; using the project .venv with Python 3.13.1.


## Baseline response

In [2]:
from pprint import pprint
from langchain_google_genai import ChatGoogleGenerativeAI

question = 'Suggest three memorable ways to explain external AI model evaluation to a non-technical conference audience.'
model = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite')
response = model.invoke(question)
pprint(response)

AIMessage(content=[{'type': 'text', 'text': 'To explain external AI evaluation to a non-technical audience, you must move away from math and code and toward **human proxies**. The goal is to demystify the "black box" by showing that evaluation is simply the act of introducing accountability.\n\nHere are three memorable metaphors to explain external AI evaluation:\n\n### 1. The "Blind Taste Test" (The Objective Benchmark)\n**The Concept:** When a company claims their AI is the "best," it’s like a chef claiming their soup is the most delicious. You can\'t just take their word for it.\n*   **The Explanation:** "Imagine two chefs—or in this case, two AI companies—each claiming their soup is the best. If we only ask the chefs to rate their own soup, they’ll always say it’s perfect. External evaluation is the \'blind taste test.\' We take their AI, put it in a neutral kitchen (a controlled testing environment), and have a group of independent judges sample it against a standard recipe. By re

In [3]:
print(response.content)

[{'type': 'text', 'text': 'To explain external AI evaluation to a non-technical audience, you must move away from math and code and toward **human proxies**. The goal is to demystify the "black box" by showing that evaluation is simply the act of introducing accountability.\n\nHere are three memorable metaphors to explain external AI evaluation:\n\n### 1. The "Blind Taste Test" (The Objective Benchmark)\n**The Concept:** When a company claims their AI is the "best," it’s like a chef claiming their soup is the most delicious. You can\'t just take their word for it.\n*   **The Explanation:** "Imagine two chefs—or in this case, two AI companies—each claiming their soup is the best. If we only ask the chefs to rate their own soup, they’ll always say it’s perfect. External evaluation is the \'blind taste test.\' We take their AI, put it in a neutral kitchen (a controlled testing environment), and have a group of independent judges sample it against a standard recipe. By removing the bias of

## Temperature experiment

In [4]:
low_temperature_model = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite', temperature=0.0)
low_temperature_response = low_temperature_model.invoke(question)
print('Temperature 0.0')
print(low_temperature_response.content)

Temperature 0.0
[{'type': 'text', 'text': 'To explain external AI evaluation to a non-technical audience, you need to move away from "metrics" and toward **trust, accountability, and quality control.**\n\nHere are three memorable analogies to help your audience grasp why we need independent, third-party testing for AI.\n\n---\n\n### 1. The "Restaurant Health Inspector" Analogy\n**The Concept:** Self-regulation vs. Independent oversight.\n\n*   **The Pitch:** "Imagine if the only person checking a restaurant’s kitchen for hygiene was the head chef. They might be a great cook, but they have a vested interest in saying everything is clean so they can keep selling food. External AI evaluation is the **Health Inspector.** They don’t work for the restaurant; they show up unannounced, look in the corners the chef ignores, and give the public an objective \'Grade A\' or \'Grade C\' rating. We need external evaluators because the companies building AI are the \'chefs\'—they are brilliant, but t

In [5]:
high_temperature_model = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite', temperature=1.0)
high_temperature_response = high_temperature_model.invoke(question)
print('Temperature 1.0')
print(high_temperature_response.content)

Temperature 1.0
[{'type': 'text', 'text': 'To explain external AI model evaluation to a non-technical audience, you need to shift the focus from "code and metrics" to "trust, accountability, and the reality of the world."\n\nHere are three memorable analogies, each targeting a different aspect of why we evaluate AI externally.\n\n---\n\n### 1. The "Independent Food Critic" Analogy (Focus: Objectivity)\n**The Concept:** Companies that build AI models are like restaurant chefs. They are incredibly proud of their creations, but they are also biased—they’ll never tell you their signature dish is too salty or that the kitchen is messy.\n\n**How to explain it:** \n"When a tech company launches a new AI model, they often tell us it’s the most delicious, healthy, and perfect meal ever cooked. But would you trust a chef to be the only person grading their own food? \n\nExternal evaluation is the **independent food critic.** We invite outside experts to sit down, taste the \'dish\' without bias,

## System-prompt experiment

In [6]:
from langchain_core.messages import HumanMessage, SystemMessage

system_instruction = (
    'You are a technical presentation coach. Give concise and accurate suggestions suitable for a three-minute academic conference presentation. ' \
    'Avoid exaggeration and explain technical concepts in accessible language.'
)
system_response = model.invoke([
    SystemMessage(content=system_instruction),
    HumanMessage(content=question),
])
print(system_response.content)

[{'type': 'text', 'text': 'To explain **external AI model evaluation** (evaluating a model using data it hasn\'t seen, rather than relying on the developer’s internal benchmarks), use these three analogies. They prioritize clarity and emphasize the necessity of independent verification.\n\n### 1. The "Standardized Test" Analogy\n*   **The Concept:** Explain that developers often train models on the same datasets they use for testing, which is like a teacher letting students see the final exam questions while they study.\n*   **The Pitch:** "Internal testing is like a teacher giving students the exact questions before the final exam—they might pass, but it doesn\'t prove they’ve mastered the subject. External evaluation is the \'Standardized Test\': we administer an exam the model has never seen before to see if it truly understands the material or if it simply memorized the answers."\n\n### 2. The "Food Critic" Analogy\n*   **The Concept:** Distinguish between a restaurant’s marketing 

## Few-shot experiment

In [7]:
from langchain_core.messages import AIMessage

few_shot_messages = [
    SystemMessage(content='Answer concisely. Begin with an analogy, then explain the technical meaning in one sentence. Avoid unsupported claims.'),
    HumanMessage(content='How can I explain a benchmark?'),
    AIMessage(content='Think of it like a standard race track. A benchmark is a fixed test used to compare systems under the same conditions.'),
    HumanMessage(content='How can I explain independent code review?'),
    AIMessage(content='Think of it like a second chef tasting a dish. Independent code review is when someone outside the original author checks code for issues and clarity.'),
    HumanMessage(content=question),
]
few_shot_response = model.invoke(few_shot_messages)
print(few_shot_response.content)

[{'type': 'text', 'text': 'Here are three memorable ways to explain external AI model evaluation:\n\n1.  **The "Blind Taste Test":** Just as independent critics review a restaurant without being involved in the cooking, external evaluation uses third-party auditors to test an AI’s performance without the bias of the company that built it.\n2.  **The "Safety Inspector":** Think of an AI developer as the car manufacturer and the external evaluator as the crash-test facility; the manufacturer builds the engine, but an independent party verifies it is safe to drive on public roads.\n3.  **The "Standardized Exam":** Just as students take a national exam to prove their knowledge, external evaluation uses a rigorous, objective set of questions—not written by the student—to certify that an AI model meets a specific quality or safety standard.', 'extras': {'signature': 'EnMKcQFpFH0ThjJB6XymYeJTElbd/tInAgy/bog0Hufh/HznXlpp3RIpBuC5ZDJZzMuQEUT2N7veEz0AwPmbW9BIkiUmhV5Gzyrnp6o2PI52JNGCZTVCQ3NAVRjrrf

## Observations

- **Difference caused by temperature**  
  Temperature changed the analogies and supporting details, but both outputs remained relatively detailed. The lower temperature felt more predictable, while the higher temperature introduced more variety.

- **Difference caused by the system prompt**  
  The system prompt made the answer more appropriate for an academic presentation. It used clearer definitions, avoided exaggerated claims and included more practical delivery guidance.

- **Difference caused by examples**  
  The examples produced the most consistent structure. Each suggestion started with an analogy and followed it with a short technical explanation.

- **Which change produced the biggest improvement**  
  The few-shot examples produced the biggest improvement because they showed the model the exact structure and level of detail I wanted, rather than only describing it.